# Notebook #12: Spatial Niche Cell-Cell Communication (spN-CCC)

Previous analysis showed that there were no shared niches between the peritoneal lesions being investigated. This notebook explores the cell-cell communication patterns within these niches to see if they share any patterns.

In [1]:
!pip install -q \
numpy==2.0.2 \
pandas==2.3.2 \
pyarrow==18.1.0 \
anndata==0.12.6 \
scanpy==1.11.5 \
squidpy \
matplotlib \
seaborn \
scikit-learn \
igraph

In [2]:
# -- Imports
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [3]:
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

drive.mount('/content/drive', force_remount = True)

Google Drive already mounted.
Mounted at /content/drive


In [4]:
# -- Paths
project_path = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

input_file = (
    project_path
    / "data"
    / "interim"
    / "spatial"
    / "10_immune_niches.h5ad"
)

output_path_results = (
    project_path
    / "results"
    / "spatial"
    / "niche_ccc")

output_path_figures = (
    project_path
    / "figures"
    / "spatial"
    / "niche_ccc")

output_path_data = (
    project_path
    / "data"
    / "interim"
    / "spatial")

for path in [
    output_path_results,
    output_path_figures,
    output_path_data,
]:
    path.mkdir(parents=True, exist_ok=True)

In [5]:
# -- Parameters
N_LRS = 10
N_PERMS = 1000
P_VALUE_CUTOFF = 0.005
MIN_SPOTS = 20
RANDOM_SEED = 3


plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.transparent"] = False

In [6]:
# -- IMPORT OBJECTS
spatial_niches = sc.read_h5ad(input_file)

required_obs_columns = {
    "sample_id",
    "immune_niche_label",
    "tissue_level_immune_class",
    "projected_senescence_score",
}

missing_obs_columns = required_obs_columns - set(spatial_niches.obs.columns)

if missing_obs_columns:
    raise KeyError(
        "Missing required obs columns: "
        f"{sorted(missing_obs_columns)}"
    )

if "q05_cell_abundance_w_sf" not in spatial_niches.obsm:
    raise KeyError(
        "Missing q05_cell_abundance_w_sf from spatial_niches.obsm."
    )

print(spatial_niches)

display(
    pd.crosstab(
        spatial_niches.obs["immune_niche_label"],
        spatial_niches.obs["sample_id"],
    )
)

AnnData object with n_obs × n_vars = 3348 × 22220
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_score', 'cluster_mean_senescence', 'projected_dysfunction_score', 'cluster_mean_dysfunction', 'immune_niche', 'im

sample_id,BEME_346,BEME_355G
immune_niche_label,,
Adaptive-enriched,0,570
Diffuse immune,0,507
Effector immune,759,0
Immune hotspot,0,87
Immune-cold,563,14
Immune-depleted,66,482
Lymphoid-rich,0,300


In [7]:
# -- Palettes & order
niche_order = [
    "Immune-depleted",
    "Immune-cold",
    "Adaptive-enriched",
    "Diffuse immune",
    "Effector immune",
    "Lymphoid-rich",
    "Immune hotspot",
]

niche_palette = {
    "Immune-depleted": "#D9D9D9",
    "Immune-cold": "#A8BED1",
    "Adaptive-enriched": "#B45E82",
    "Diffuse immune": "#6F9272",
    "Effector immune": "#A9643A",
    "Lymphoid-rich": "#CDA03A",
    "Immune hotspot": "#8E1F2F",
}

celltype_order = [
    "Mono-C",
    "Mono-NC",
    "TRM",
    "DC",
    "NK-CD16+",
    "NK-CD16-",
    "CD4 T",
    "CD8 T",
    "Treg",
    "γδ T",
    "B",
]

immune_palette = {
    "Mono-C": "#E07B54",
    "Mono-NC": "#F2A65A",
    "TRM": "#C14F3A",
    "DC": "#7B4FA6",
    "NK-CD16+": "#2D8C6E",
    "NK-CD16-": "#6DBF9E",
    "CD4 T": "#0B4A7E",
    "CD8 T": "#2E6FA3",
    "Treg": "#5B9EC9",
    "γδ T": "#A8CBE0",
    "B": "#E8C84A",
}

observed_niches = (
    spatial_niches.obs["immune_niche_label"]
    .dropna()
    .astype(str)
    .unique()
)

unexpected_niches = sorted(set(observed_niches) - set(niche_order))

if unexpected_niches:
    raise ValueError(
        "Unexpected niche labels found: "
        f"{unexpected_niches}"
    )

## Assign a dominant inferred immune population per spot

Dominant population is assigned from rank-scaled cell2location abundances. Immune-depleted spots are excluded from assignment because a dominant immune label would be low-confidence there.


In [8]:
abundance = spatial_niches.obsm["q05_cell_abundance_w_sf"].copy()

abundance.columns = [
    column.split("_sf_", 1)[-1]
    for column in abundance.columns
]

available_cell_types = [
    cell_type
    for cell_type in celltype_order
    if cell_type in abundance.columns
]

missing_cell_types = [
    cell_type
    for cell_type in celltype_order
    if cell_type not in abundance.columns
]

if missing_cell_types:
    print(
        "Cell types not found in abundance matrix:",
        missing_cell_types,
    )

if not available_cell_types:
    raise ValueError(
        "No expected immune-cell abundance columns were found."
    )

abundance = abundance[available_cell_types]
ranked_abundance = abundance.rank(pct=True)

immune_spots = (
    spatial_niches.obs["immune_niche_label"].astype(str)
    != "Immune-depleted"
)

spatial_niches.obs["dominant_celltype"] = "Non-immune / low confidence"

spatial_niches.obs.loc[
    immune_spots,
    "dominant_celltype",
] = ranked_abundance.loc[immune_spots].idxmax(axis=1)

dominant_order = [
    cell_type
    for cell_type in celltype_order
    if cell_type in (
        spatial_niches.obs["dominant_celltype"]
        .astype(str)
        .unique()
    )
]

dominant_order += ["Non-immune / low confidence"]

spatial_niches.obs["dominant_celltype"] = pd.Categorical(
    spatial_niches.obs["dominant_celltype"],
    categories=dominant_order,
    ordered=True,
)

display(
    pd.crosstab(
        spatial_niches.obs["immune_niche_label"],
        spatial_niches.obs["dominant_celltype"],
    )
)

dominant_celltype,Mono-C,Mono-NC,TRM,DC,NK-CD16+,NK-CD16-,CD4 T,CD8 T,Treg,γδ T,B,Non-immune / low confidence
immune_niche_label,,,,,,,,,,,,
Adaptive-enriched,3,1,6,1,3,6,411,0,0,126,13,0
Diffuse immune,6,2,30,16,62,39,105,0,0,221,26,0
Effector immune,84,106,10,112,1,1,101,33,310,1,0,0
Immune hotspot,13,2,19,11,17,12,0,2,0,11,0,0
Immune-cold,136,46,12,10,10,5,236,15,83,4,20,0
Immune-depleted,0,0,0,0,0,0,0,0,0,0,0,548
Lymphoid-rich,18,5,19,17,15,15,70,0,0,54,87,0


In [9]:
# -- Run sq LR analysis within each niche
results_by_niche = {}

for niche in niche_order:

    if niche not in observed_niches:
        continue

    subset = spatial_niches[
        (
            spatial_niches.obs["immune_niche_label"].astype(str)
            == niche
        )
        & (
            spatial_niches.obs["dominant_celltype"].astype(str)
            != "Non-immune / low confidence"
        )
    ].copy()

    subset.obs["dominant_celltype"] = (
        subset.obs["dominant_celltype"]
        .cat.remove_unused_categories()
    )

    n_cell_types = subset.obs["dominant_celltype"].nunique()

    if subset.n_obs < MIN_SPOTS:
        print(
            f"Skipping {niche}: only {subset.n_obs} eligible spots."
        )
        continue

    if n_cell_types < 2:
        print(
            f"Skipping {niche}: only {n_cell_types} dominant cell type."
        )
        continue

    print(
        f"Running ligrec for {niche}: "
        f"{subset.n_obs} spots, "
        f"{n_cell_types} cell types"
    )

    sq.gr.ligrec(
        subset,
        cluster_key="dominant_celltype",
        n_perms=N_PERMS,
        alpha=P_VALUE_CUTOFF,
        remove_nonsig_interactions=True,
        remove_empty_interactions=True,
        seed=RANDOM_SEED,
        n_jobs=1,
        copy=False,
        use_raw=False,
    )

    results_by_niche[niche] = (
        subset.uns["dominant_celltype_ligrec"]
    )

print(
    "Completed niches:",
    list(results_by_niche),
)

Skipping Immune-depleted: only 0 eligible spots.
Running ligrec for Immune-cold: 577 spots, 11 cell types


  0%|          | 0/1000 [00:00<?, ?permutation/s]

Running ligrec for Adaptive-enriched: 570 spots, 9 cell types


  0%|          | 0/1000 [00:00<?, ?permutation/s]

Running ligrec for Diffuse immune: 507 spots, 9 cell types


...

Running ligrec for Lymphoid-rich: 300 spots, 9 cell types


  0%|          | 0/1000 [00:00<?, ?permutation/s]

Running ligrec for Immune hotspot: 87 spots, 8 cell types


  0%|          | 0/1000 [00:00<?, ?permutation/s]

Completed niches: ['Immune-cold', 'Adaptive-enriched', 'Diffuse immune', 'Effector immune', 'Lymphoid-rich', 'Immune hotspot']


In [10]:
# Save results
def top_n_ligrec(
    result,
    n=N_LRS,
):
    """Return the top interactions ranked by minimum p-value."""

    pvalues = result["pvalues"]
    means = result["means"]

    significant = means.where(
    pvalues <= P_VALUE_CUTOFF)

    top_interactions = (
        significant
        .max(axis=1)
        .sort_values(ascending=False)
        .head(10)
        .index
        )


    top_result = {
        "means": means.loc[top_interactions],
        "pvalues": pvalues.loc[top_interactions],
    }

    if "metadata" in result:
        top_result["metadata"] = (
            result["metadata"].loc[top_interactions]
        )

    return top_result


def safe_filename(label):
    return (
        str(label)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("–", "-")
    )


def save_ligrec_by_niche(
    results,
    n=N_LRS,
    alpha=P_VALUE_CUTOFF,
):
    """Save niche-specific Squidpy ligrec figures and tables."""

    for niche, result in results.items():

        print(
            f"Saving ligrec results: {niche}"
        )

        top_result = top_n_ligrec(
            result,
            n=n,
        )

        filename = safe_filename(niche)

        sq.pl.ligrec(
            top_result,
            cluster_key="dominant_celltype",
            alpha=alpha,
            remove_nonsig_interactions=True,
            show=True,
            title="",
            figsize=(16, 10),
            show_size_legend=True,
            show_colorbar=True,
        )

        plt.savefig(
            output_path_figures
            / f"{filename}_ligrec.png",
            dpi=300,
            )

        plt.close()

        top_result["means"].to_csv(
            output_path_results
            / f"{filename}_ligrec_means.csv"
        )

        top_result["pvalues"].to_csv(
            output_path_results
            / f"{filename}_ligrec_pvalues.csv"
        )


save_ligrec_by_niche(
    results_by_niche,
    n=N_LRS,
    alpha=P_VALUE_CUTOFF,
)

Saving ligrec results: Immune-cold
Saving ligrec results: Adaptive-enriched
Saving ligrec results: Diffuse immune
Saving ligrec results: Effector immune
Saving ligrec results: Lymphoid-rich
Saving ligrec results: Immune hotspot


In [11]:
# Incoming and Outgoing Communication Scores
def calculate_ccc_scores_by_niche(
    results,
    p_value_cutoff=P_VALUE_CUTOFF,
):
    """Summarize significant incoming and outgoing communication per cell type."""

    all_scores = []

    for niche, result in results.items():

        means = result["means"].copy()
        pvalues = result["pvalues"].copy()

        significant_means = (
            means
            .where(pvalues <= p_value_cutoff)
            .dropna(how="all")
        )

        if significant_means.empty:
            print(
                f"No significant CCC scores for {niche}."
            )
            continue

        pair_scores = (
            significant_means
            .sum(axis=0)
            .rename("communication_score")
            .reset_index()
        )

        pair_scores.columns = [
            "source",
            "target",
            "communication_score",
        ]

        outgoing = (
            pair_scores
            .groupby("source", observed=True)["communication_score"]
            .sum()
            .rename("outgoing_score")
            .reset_index()
            .rename(columns={"source": "cell_type"})
        )

        incoming = (
            pair_scores
            .groupby("target", observed=True)["communication_score"]
            .sum()
            .rename("incoming_score")
            .reset_index()
            .rename(columns={"target": "cell_type"})
        )

        scores = (
            outgoing
            .merge(
                incoming,
                on="cell_type",
                how="outer",
            )
            .fillna(0)
        )

        scores["niche"] = niche
        all_scores.append(scores)

    if not all_scores:
        return pd.DataFrame(
            columns=[
                "cell_type",
                "outgoing_score",
                "incoming_score",
                "niche",
            ]
        )

    return pd.concat(
        all_scores,
        ignore_index=True,
    )


ccc_scores = calculate_ccc_scores_by_niche(
    results_by_niche
)

ccc_scores.to_csv(
    output_path_results / "12_niche_ccc_scores.csv",
    index=False,
)

display(ccc_scores.head())

,cell_type,outgoing_score,incoming_score,niche
0,B,7639.740258,4779.586894,Immune-cold
1,CD4 T,1625.257727,2975.244853,Immune-cold
2,CD8 T,487.431986,1330.174049,Immune-cold
3,DC,1451.320009,2688.080508,Immune-cold
4,Mono-C,1277.742015,3030.314821,Immune-cold


In [12]:
# -- Add projected SEN scores
def calculate_senescence_by_niche(
    adata,
    niche_key="immune_niche_label",
    cell_type_key="dominant_celltype",
    senescence_key="projected_senescence_score",
):
    """Calculate mean projected senescence for each cell type within each niche."""

    valid_spots = (
        adata.obs[cell_type_key].astype(str)
        != "Non-immune / low confidence"
    )

    return (
        adata.obs.loc[valid_spots]
        .groupby(
            [niche_key, cell_type_key],
            observed=True,
        )[senescence_key]
        .mean()
        .rename("mean_senescence_score")
        .reset_index()
        .rename(
            columns={
                niche_key: "niche",
                cell_type_key: "cell_type",
            }
        )
    )


senescence_scores = calculate_senescence_by_niche(
    spatial_niches
)

ccc_senescence = ccc_scores.merge(
    senescence_scores,
    on=["niche", "cell_type"],
    how="left",
)

ccc_senescence.to_csv(
    output_path_results
    / "12_niche_ccc_senescence_scores.csv",
    index=False,
)

display(ccc_senescence.head())

,cell_type,outgoing_score,incoming_score,niche,mean_senescence_score
0,B,7639.740258,4779.586894,Immune-cold,-0.083015
1,CD4 T,1625.257727,2975.244853,Immune-cold,-0.083183
2,CD8 T,487.431986,1330.174049,Immune-cold,-0.090370
3,DC,1451.320009,2688.080508,Immune-cold,-0.079106
4,Mono-C,1277.742015,3030.314821,Immune-cold,-0.069174


In [13]:
# -- LR activity vs. SEN
def plot_senescence_communication(
    data,
    communication_col,
    y_label,
    output_file,
):
    fig, ax = plt.subplots(figsize=(11, 6))

    sns.scatterplot(
        data=data,
        x="mean_senescence_score",
        y=communication_col,
        hue="niche",
        hue_order=[
            niche
            for niche in niche_order
            if niche in data["niche"].unique()
        ],
        palette=niche_palette,
        style="cell_type",
        style_order=[
            cell_type
            for cell_type in celltype_order
            if cell_type in data["cell_type"].unique()
        ],
        s=120,
        ax=ax,
    )

    ax.set_xlabel(
        "Mean projected senescence score"
    )
    ax.set_ylabel(y_label)
    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()
    plt.savefig(
        output_path_figures / output_file,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()


plot_senescence_communication(
    ccc_senescence,
    communication_col="outgoing_score",
    y_label="Outgoing communication score",
    output_file="12_outgoing_communication_by_senescence.png",
)

plot_senescence_communication(
    ccc_senescence,
    communication_col="incoming_score",
    y_label="Incoming communication score",
    output_file="12_incoming_communication_by_senescence.png",
)

In [14]:
# Standardize with Z-score

def add_within_niche_zscores(
    data
):
    """Add within-niche z-scores for communication and senescence metrics."""

    plot_data = data.copy()

    columns_to_standardize = [
        "outgoing_score",
        "incoming_score",
        "mean_senescence_score"
    ]

    for column in columns_to_standardize:

        if column not in plot_data.columns:
            continue

        # -- Convert sparse arrays to regular dense numeric values
        if isinstance(
            plot_data[column].dtype,
            pd.SparseDtype
        ):
            plot_data[column] = (
                plot_data[column]
                .sparse
                .to_dense()
            )

        plot_data[column] = pd.to_numeric(
            plot_data[column],
            errors="coerce"
        )

        plot_data[
            f"{column}_z"
        ] = (
            plot_data
            .groupby(
                "niche",
                observed=True
            )[column]
            .transform(
                lambda values: (
                    (
                        values
                        - values.mean()
                    )
                    / values.std(
                        ddof=0
                    )
                )
                if values.std(
                    ddof=0
                ) > 0
                else 0.0
            )
        )

    return plot_data


def plot_niche_facets(
    data,
    x,
    y,
    x_label,
    y_label,
    output_file,
):
    observed_niches = [
        niche
        for niche in niche_order
        if niche in data["niche"].unique()
    ]

    observed_cell_types = [
        cell_type
        for cell_type in celltype_order
        if cell_type in data["cell_type"].unique()
    ]

    facet = sns.FacetGrid(
        data,
        col="niche",
        col_order=observed_niches,
        sharex=False,
        sharey=False,
    )

    facet.map_dataframe(
        sns.scatterplot,
        x=x,
        y=y,
        hue="cell_type",
        hue_order=observed_cell_types,
        palette=immune_palette,
        s=100,
    )

    for axis in facet.axes.flat:
        axis.axvline(0, linestyle="--", linewidth=1)
        axis.axhline(0, linestyle="--", linewidth=1)
        axis.grid(True, linestyle="--", alpha=0.4)

    facet.add_legend()
    facet.set_axis_labels(x_label, y_label)

    plt.savefig(
        output_path_figures / output_file,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()


plot_data = add_within_niche_zscores(
    ccc_senescence
)

# -- Incoming versus outgoing communication
plot_niche_facets(
    plot_data,
    x="incoming_score_z",
    y="outgoing_score_z",
    x_label="Incoming communication score (z-score)",
    y_label="Outgoing communication score (z-score)",
    output_file="12_incoming_vs_outgoing_by_niche.png",
)

# -- Projected senescence versus incoming communication
plot_niche_facets(
    plot_data,
    x="mean_senescence_score_z",
    y="incoming_score_z",
    x_label="Projected senescence score (z-score)",
    y_label="Incoming communication score (z-score)",
    output_file="12_senescence_vs_incoming_by_niche.png",
)

# -- Projected senescence versus outgoing communication
plot_niche_facets(
    plot_data,
    x="mean_senescence_score_z",
    y="outgoing_score_z",
    x_label="Projected senescence score (z-score)",
    y_label="Outgoing communication score (z-score)",
    output_file="12_senescence_vs_outgoing_by_niche.png",
)

## Optional lesion-specific niche comparison

The current primary analysis pools spots sharing a niche identity across lesions. Before running lesion-specific ligrec, inspect the sample × niche counts because several combinations may contain too few spots or dominant populations.


In [15]:
sample_niche_counts = pd.crosstab(
    spatial_niches.obs["immune_niche_label"],
    spatial_niches.obs["sample_id"],
)

display(sample_niche_counts)

sample_niche_counts.to_csv(
    output_path_results
    / "12_sample_niche_spot_counts.csv"
)

sample_id,BEME_346,BEME_355G
immune_niche_label,,
Adaptive-enriched,0,570
Diffuse immune,0,507
Effector immune,759,0
Immune hotspot,0,87
Immune-cold,563,14
Immune-depleted,66,482
Lymphoid-rich,0,300


In [16]:
output_file = (
    output_path_data
    / "12_spatial_niche_ccc.h5ad"
)

spatial_niches.write_h5ad(output_file)

saved = sc.read_h5ad(
    output_file,
    backed="r",
)

print(saved)
saved.file.close()

AnnData object with n_obs × n_vars = 3348 × 22220 backed at '/content/drive/MyDrive/endo-immune-atlas/data/interim/spatial/12_spatial_niche_ccc.h5ad'
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_score', 'clus